In [ ]:
!pip install hyperopt

In [ ]:
!pip install hpsklearn

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 5.6 MB/s eta 0:00:00
  Created wheel for hpsklearn: filename=hpsklearn-0.1.0-py3-none-any.whl size=23916 sha256=43b8a2e443b16006711a354ed4890da9cec31f26c8a7a648d3cbea25662751a1
  Stored in directory: /root/.cache/pip/wheels/08/6b/8c/977f9f05e977944f6ef84184cf00dd08d059eb04dfe0c1fb9c
Successfully built hpsklearn


In [ ]:
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import r2_score
from xgboost import XGBRegressor
from hyperopt import space_eval
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import mean_absolute_percentage_error

In [ ]:
df = pd.read_csv('/content/cars_data_cleaned1_post_model_selection.csv')

In [ ]:
df = df.drop(columns = ['region'])

In [ ]:
df.isnull().sum()

,0
price,0
manufacturer,0
model,0
cylinders,0
fuel,0
odometer,0
transmission,0
drive,0
type,0
seats,0


In [ ]:
X = df.drop(columns = ['price'])
y = df['price']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

In [ ]:
columns_to_oe = ['manufacturer', 'model']
columns_to_ohe = ['fuel', 'drive', 'transmission', 'type']

In [ ]:
preprocessor = ColumnTransformer(
    transformers = [
        ('num', StandardScaler(), ['cylinders', 'odometer', 'seats', 'engine_cc', 'max_power', 'age']),
        ('cat1', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), columns_to_oe),
        ('cat2', OneHotEncoder(handle_unknown= 'ignore'), columns_to_ohe)
    ]
)

In [ ]:
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from sklearn.metrics import r2_score

# Define the search space for RandomForestRegressor
space = {
    'n_estimators': hp.quniform('n_estimators', 100, 600, 100), # Number of trees
    'max_depth': hp.choice('max_depth', [None, 10, 20, 30]),    # Maximum depth of the tree
    'min_samples_split': hp.quniform('min_samples_split', 2, 10, 1), # Minimum number of samples required to split an internal node
    'min_samples_leaf': hp.quniform('min_samples_leaf', 1, 5, 1),   # Minimum number of samples required to be at a leaf node
    'max_features': hp.choice('max_features', ['sqrt', 'log2', 0.8, 1.0]) # Number of features to consider when looking for the best split
}

# Objective function to minimize (negative R2 score)
def objective(params):
    n_estimators = int(params['n_estimators'])
    min_samples_split = int(params['min_samples_split'])
    min_samples_leaf = int(params['min_samples_leaf'])

    model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=params['max_depth'],
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=params['max_features'],
        n_jobs=-1, # Use all available cores
        random_state=42
    )

    # Create a pipeline with the existing preprocessor and the current model
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    # Hyperopt minimizes the objective function, so we return negative R2
    score = r2_score(y_test, y_pred)

    return {'loss': -score, 'status': STATUS_OK, 'params': params}

# Run the optimization
trials = Trials()
best = fmin(fn=objective,
            space=space,
            algo=tpe.suggest,
            max_evals=50, # Number of different hyperparameter combinations to try
            trials=trials,
            rstate=np.random.default_rng(42) # For reproducibility
           )

print("Best parameters found:", best)
# took 18 min

100%|██████████| 50/50 [22:03<00:00, 26.47s/trial, best loss: -0.9182306105090914]
Best parameters found: {'max_depth': np.int64(2), 'max_features': np.int64(1), 'min_samples_leaf': np.float64(1.0), 'min_samples_split': np.float64(2.0), 'n_estimators': np.float64(300.0)}


In [ ]:
from hyperopt import space_eval

best_params = space_eval(space, best)
print("Best actual parameters:", best_params)

# Train the final model with the best parameters
final_rf_model = RandomForestRegressor(
    n_estimators=int(best_params['n_estimators']),
    max_depth=best_params['max_depth'],
    min_samples_split=int(best_params['min_samples_split']),
    min_samples_leaf=int(best_params['min_samples_leaf']),
    max_features=best_params['max_features'],
    n_jobs=-1,
    random_state=42
)

final_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', final_rf_model)
])

final_pipeline.fit(X_train, y_train)
y_pred_final = final_pipeline.predict(X_test)

print("\nR2 score with best parameters:", r2_score(y_test, y_pred_final)) # R2 score with best parameters: 0.9342636775370158
print("MAE with best parameters:", mean_absolute_error(y_test, y_pred_final)) # MAE with best parameters: 0.11522815297424159
# Best actual parameters: {'max_depth': 30, 'max_features': 'sqrt', 'min_samples_leaf': 1.0, 'min_samples_split': 3.0, 'n_estimators': 500.0}

NameError: name 'space' is not defined

In [ ]:
rf_model = RandomForestRegressor(max_depth = 30, max_features = 'sqrt', min_samples_leaf = 1, min_samples_split = 3, n_estimators = 500, n_jobs = -1, random_state = 42)

In [ ]:
rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', rf_model)
])

In [ ]:
rf_pipeline.fit(X_train, y_train)
y_pred = rf_pipeline.predict(X_test)

In [ ]:
print("\nR2 score with best parameters for random forest regressor:", r2_score(y_test, y_pred))
print("MAE with best parameters for random forest regressor :", mean_absolute_error(y_test, y_pred))
print("MAPE with best parameters for random forest regressor is: ", mean_absolute_percentage_error(y_test, y_pred))


R2 score with best parameters for random forest regressor: 0.9167674814980397
MAE with best parameters for random forest regressor : 101250.62168898826
MAPE with best parameters for random forest regressor is:  0.171297539870755


# **XgBoost scoring with hyperparameter tuning**

In [ ]:
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials

# Define the search space for RandomForestRegressor
space = {
    'n_estimators': hp.quniform('n_estimators', 100, 200, 300),
    'max_depth': hp.quniform('max_depth', 3, 6, 9),
    'learning_rate': hp.loguniform('learning_rate', np.log(0.01), np.log(0.3)),
    'subsample': hp.uniform('subsample', 0.6, 1.0),
    'colsample_bytree': hp.uniform('colsample_bytree', 0.6, 1.0),
    'gamma': hp.uniform('gamma', 0, 5),
    'reg_alpha': hp.uniform('reg_alpha', 0, 5),
    'reg_lambda': hp.uniform('reg_lambda', 0, 5)
}

# Objective function to minimize (negative R2 score)
def objective(params):

    model = XGBRegressor(
        n_estimators=int(params['n_estimators']),
        max_depth=int(params['max_depth']),
        learning_rate=params['learning_rate'],
        subsample=params['subsample'],
        colsample_bytree=params['colsample_bytree'],
        gamma=params['gamma'],
        reg_alpha=params['reg_alpha'],
        reg_lambda=params['reg_lambda'],
        n_jobs=-1,
        random_state=42,
        verbosity=0
    )

    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    score = r2_score(y_test, y_pred)

    return {'loss': -score, 'status': STATUS_OK}

trials = Trials()
best = fmin(
    fn=objective,
    space=space,
    algo=tpe.suggest,
    max_evals=50,
    trials=trials,
    rstate=np.random.default_rng(42)
)

print("Best parameters:", best)

100%|██████████| 50/50 [04:10<00:00,  5.02s/trial, best loss: -0.918734610080719]
Best parameters: {'colsample_bytree': np.float64(0.738443279994232), 'gamma': np.float64(4.2882345233149906), 'learning_rate': np.float64(0.06551361079171157), 'max_depth': np.float64(9.0), 'n_estimators': np.float64(300.0), 'reg_alpha': np.float64(1.3416237685526877), 'reg_lambda': np.float64(0.20630178685456357), 'subsample': np.float64(0.6138813743427279)}


In [ ]:
final_xgb_model = XGBRegressor(
    n_estimators=int(best['n_estimators']),
    max_depth=int(best['max_depth']),
    learning_rate=best['learning_rate'],
    subsample=best['subsample'],
    colsample_bytree=best['colsample_bytree'],
    gamma=best['gamma'],
    reg_alpha=best['reg_alpha'],
    reg_lambda=best['reg_lambda'],
    n_jobs=-1,
    random_state=42,
    verbosity=0
)

final_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', final_xgb_model)
])

final_pipeline.fit(X_train, y_train)

y_pred_final = final_pipeline.predict(X_test)

print("\nR2 score:", r2_score(y_test, y_pred_final))
print("MAE:", mean_absolute_error(y_test, y_pred_final))


R2 score: 0.918734610080719
MAE: 99767.84375


In [ ]:
print("MAPE with best parameters for xgboost regressor is: ", mean_absolute_percentage_error(y_test, y_pred_final))

MAPE with best parameters for xgboost regressor is:  0.16592548787593842


# **Extra Trees with hyperparameter tuning**

In [ ]:
space = {
    'n_estimators': hp.quniform('n_estimators', 100, 700, 50),
    'max_depth': hp.choice('max_depth', [None, 10, 20, 30, 40]),
    'min_samples_split': hp.quniform('min_samples_split', 2, 10, 1),
    'min_samples_leaf': hp.quniform('min_samples_leaf', 1, 5, 1),
    'max_features': hp.choice('max_features', ['sqrt', 'log2', 0.8, 1.0]),
    'bootstrap': hp.choice('bootstrap', [True, False])
}

def objective(params):

    model = ExtraTreesRegressor(
        n_estimators=int(params['n_estimators']),
        max_depth=params['max_depth'],
        min_samples_split=int(params['min_samples_split']),
        min_samples_leaf=int(params['min_samples_leaf']),
        max_features=params['max_features'],
        bootstrap=params['bootstrap'],
        n_jobs=-1,
        random_state=42
    )

    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    score = r2_score(y_test, y_pred)

    return {'loss': -score, 'status': STATUS_OK}

trials = Trials()

best = fmin(
    fn=objective,
    space=space,
    algo=tpe.suggest,
    max_evals=50,
    trials=trials,
    rstate=np.random.default_rng(42)
)

print("Best parameters:", best)
# took 25 min

100%|██████████| 50/50 [13:52<00:00, 16.64s/trial, best loss: -0.9198919411645228]
Best parameters: {'bootstrap': np.int64(1), 'max_depth': np.int64(2), 'max_features': np.int64(2), 'min_samples_leaf': np.float64(1.0), 'min_samples_split': np.float64(6.0), 'n_estimators': np.float64(700.0)}


In [ ]:
best_params = space_eval(space, best)
print("Best actual parameters:", best_params)

Best actual parameters: {'bootstrap': False, 'max_depth': 20, 'max_features': 0.8, 'min_samples_leaf': 1.0, 'min_samples_split': 6.0, 'n_estimators': 700.0}


In [ ]:
final_et_model = ExtraTreesRegressor(
    n_estimators=int(best_params['n_estimators']),
    max_depth=best_params['max_depth'],
    min_samples_split=int(best_params['min_samples_split']),
    min_samples_leaf=int(best_params['min_samples_leaf']),
    max_features=best_params['max_features'],
    bootstrap=best_params['bootstrap'],
    n_jobs=-1,
    random_state=42
)

final_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', final_et_model)
])

final_pipeline.fit(X_train, y_train)

y_pred_final2 = final_pipeline.predict(X_test)

print("\nR2 score:", r2_score(y_test, y_pred_final2))
print("MAE:", mean_absolute_error(y_test, y_pred_final2))


R2 score: 0.9198919411645228
MAE: 98257.64335341593


In [ ]:
print("MAPE with best parameters for extra trees regressor is: ", mean_absolute_percentage_error(y_test, y_pred_final2))

MAPE with best parameters for extra trees regressor is:  0.16358250336418892


In [ ]:
df[df['transmission'] == 'automatic']['model'].value_counts()

,count
model,
city,697
creta,491
verna,368
celerio,349
c-class,340
...,...
lx,1
h3,1
phantom,1
